# Part 3 - Logistic Regression
**Gray Interface '26 | Task 2**

Dataset: [Santander Customer Transaction Prediction](https://www.kaggle.com/competitions/santander-customer-transaction-prediction)

## Importing the dependencies

In [ ]:
!pip install kagglehub --quiet


In [ ]:
import kagglehub
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             roc_curve, classification_report)


## Loading the Dataset

In [ ]:
path = kagglehub.dataset_download("competitions/santander-customer-transaction-prediction")
# Load training data
df = pd.read_csv(os.path.join(path, "train.csv"))

print("Shape:", df.shape)
df.head()


## Exploratory Data Analysis

In [ ]:
# Check class balance — important for classification tasks
class_counts = df['target'].value_counts()
print("Class distribution:")
print(class_counts)
print(f"
Class imbalance ratio: {class_counts[0]/class_counts[1]:.1f}:1")


In [ ]:
# Visualize class imbalance
plt.figure(figsize=(6, 4))
plt.bar(['Not Transacted (0)', 'Transacted (1)'], class_counts.values,
        color=['steelblue', '#E50914'])
plt.title('Target Class Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
# Check for missing values
print("Missing values:", df.isnull().sum().sum())
print("Data types:", df.dtypes.value_counts())


## Preprocessing

In [ ]:
# Drop ID column — not a feature
df = df.drop(columns=['ID_code'])

# Separate features and target
X = df.drop(columns=['target'])
y = df['target']

# Train-test split (80/20), stratified to preserve class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features — Logistic Regression is sensitive to feature magnitude
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")
print(f"Train class ratio: {y_train.value_counts().to_dict()}")


## Training Logistic Regression

In [ ]:
# Train baseline Logistic Regression
# class_weight='balanced' tells the model to compensate for class imbalance
# by giving more importance to the minority class (target=1)
lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42,
                               class_weight='balanced', solver='lbfgs')
lr_model.fit(X_train_scaled, y_train)
print("Model trained.")


## Evaluation

In [ ]:
def evaluate_classifier(model, X_, y_, label):
    preds      = model.predict(X_)
    preds_prob = model.predict_proba(X_)[:, 1]

    print(f"\n=== {label} ===")
    print(f"Accuracy  : {accuracy_score(y_, preds):.4f}")
    print(f"Precision : {precision_score(y_, preds):.4f}")
    print(f"Recall    : {recall_score(y_, preds):.4f}")
    print(f"F1 Score  : {f1_score(y_, preds):.4f}")
    print(f"ROC-AUC   : {roc_auc_score(y_, preds_prob):.4f}")
    return preds, preds_prob

train_preds, train_probs = evaluate_classifier(lr_model, X_train_scaled, y_train, "Training Set")
test_preds,  test_probs  = evaluate_classifier(lr_model, X_test_scaled,  y_test,  "Test Set")


## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, test_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: 0', 'Pred: 1'],
            yticklabels=['True: 0', 'True: 1'])
plt.title('Confusion Matrix (Test Set)')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, test_preds))


## ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_probs)
auc_score   = roc_auc_score(y_test, test_probs)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#E50914', linewidth=2, label=f'ROC Curve (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression')
plt.legend()
plt.tight_layout()
plt.show()


## Experimenting with Different Values of C

In [ ]:
# C is the inverse of regularization strength
# Small C = strong regularization (simpler model)
# Large C = weak regularization (model fits training data more closely)

C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
results = []

for c in C_values:
    model = LogisticRegression(C=c, max_iter=1000, random_state=42,
                                class_weight='balanced', solver='lbfgs')
    model.fit(X_train_scaled, y_train)
    preds_prob = model.predict_proba(X_test_scaled)[:, 1]
    preds      = model.predict(X_test_scaled)
    results.append({
        'C': c,
        'Accuracy':  round(accuracy_score(y_test, preds), 4),
        'Recall':    round(recall_score(y_test, preds), 4),
        'F1':        round(f1_score(y_test, preds), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, preds_prob), 4),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


In [ ]:
# Plot ROC-AUC vs C
plt.figure(figsize=(8, 5))
plt.semilogx(results_df['C'], results_df['ROC-AUC'], marker='o', color='steelblue', linewidth=2)
plt.title('ROC-AUC vs Regularization Parameter C')
plt.xlabel('C (log scale)')
plt.ylabel('ROC-AUC')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Observations

### Feature Scaling
Feature scaling is essential here. The 200 features have different ranges — without scaling, features with larger magnitudes dominate the gradient updates and the model converges poorly or not at all.

### Regularization (C parameter)
- Very small C (e.g. 0.001): Too much regularization — model is too simple, underfits, misses positive cases.
- C = 1.0 (default): Good balance between bias and variance.
- Very large C (e.g. 100): Almost no regularization — model may overfit to training data.
- The ROC-AUC vs C plot shows the sweet spot — typically around C = 0.1 to 1.0 for this dataset.

### Class Imbalance
The dataset is heavily imbalanced (~10:1 ratio of non-transactions to transactions). Without `class_weight='balanced'`:
- The model would predict class 0 for almost everything and still get ~90% accuracy.
- But Recall and F1 for the positive class would be near zero — useless for the actual task.
- `class_weight='balanced'` penalizes misclassifying the minority class more, pushing the model to actually learn it.

### Best Metric for This Problem
Accuracy is misleading here due to imbalance. **ROC-AUC** and **Recall** are the meaningful metrics — we care about correctly identifying customers who *will* transact.
